In [1]:
import requests
import pandas as pd
import datetime
import json
import sqlite3
import traceback

# === 設定 ===
# DB_FILE = "/Users/lulutsai/Documents/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
LOG_FILE = "eb_evaluation.log"
DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
API_TOKEN = 'sk-a94e722134454665b8fef72ddb349f37'
API_URL = "http://localhost:3000/api/v1/chat/completions"
LIMIT_COUNT = 0 # ← 你想一次處理幾筆資料（可自行調整）
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}

# === 🔹 Log 工具 ===
def write_log(message: str, link_id=None, article_id=None, comment_id=None, status=None):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    id_info = []
    if link_id:
        id_info.append(f"link_id={link_id}")
    if article_id:
        id_info.append(f"article_id={article_id}")
    if comment_id:
        id_info.append(f"comment_id={comment_id}")
    if status:
        id_info.append(f"status={status}")
    id_str = " | ".join(id_info)
    line = f"[{timestamp}] {message}"
    if id_str:
        line += f" | {id_str}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)


# === 🔹 確保欄位存在 ===
def ensure_column(conn, table, column, definition):
    cur = conn.cursor()
    cur.execute(f"PRAGMA table_info({table});")
    columns = [c[1] for c in cur.fetchall()]
    if column not in columns:
        cur.execute(f"ALTER TABLE {table} ADD COLUMN {column} {definition};")
        conn.commit()
        write_log(f"🧱 已新增欄位 {table}.{column}")
    else:
        write_log(f"🔎 欄位 {column} 已存在，略過")


# === 🔹 建立 eb_evaluation 資料表 ===
def ensure_eb_evaluation_table(conn):
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS eb_evaluation (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        link_id TEXT NOT NULL,
        article_id TEXT NOT NULL,
        comment_id TEXT,
        level INTEGER DEFAULT 1,
        created_at TEXT,
        strategy_power REAL,
        strategy_emotion REAL,
        strategy_blame REAL,
        score_fear REAL,
        score_obligation REAL,
        score_guilt REAL,
        pua_source TEXT,
        main_strategy TEXT,
        main_strategy_detail TEXT,
        confidence REAL,
        score_overall REAL,
        model_name TEXT,
        model_version TEXT,
        knowledge_base TEXT,
        FOREIGN KEY (article_id) REFERENCES articles(id),
        FOREIGN KEY (link_id) REFERENCES links(id)
    );
    """)
    conn.commit()
    write_log("✅ 確認資料表 eb_evaluation 已存在")


# === 🔹 取得待分析文章 ===
def get_pending_articles(conn, limit):
    query = f"""
    SELECT id, link_id, content
    FROM articles
    WHERE content IS NOT NULL
      AND TRIM(content) != ''
      AND (pua_status IS NULL OR pua_status = 'pending')
    LIMIT {limit};
    """
    df = pd.read_sql_query(query, conn)
    write_log(f"📘 共讀取 {len(df)} 筆文章待分析（上限 {limit}）")
    return df


# === 🔹 更新文章狀態 ===
def update_article_status(conn, article_id, status, reason=None):
    cur = conn.cursor()
    cur.execute(
        "UPDATE articles SET pua_status = ?, pua_status_reason = ? WHERE id = ?;",
        (status, reason, article_id)
    )
    conn.commit()


# === 🔹 呼叫模型分析 ===
def analyze_text(text):
    data = {
        "model": "mistral:latest",
        "knowledge": ["pua_db"],
        "messages": [
            {
                "role": "system",
                "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
  "strategy_power": 0~5,
  "strategy_emotion": 0~5,
  "strategy_blame": 0~5,
  "score_fear": 0~5,
  "score_obligation": 0~5,
  "score_guilt": 0~5,
  "pua_source": "family／partner／friend／workplace／online／self",
  "main_strategy": "power／emotion／blame／none",
  "main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
  "confidence": 0~1,
  "score_overall": 0~5
}

定義說明：
- 0 代表該特徵完全沒有出現；
- 5 代表該特徵非常明顯；
- main_strategy 僅能是 power、emotion、blame、none 四類；
- 所有分數均為數值，請勿包含文字說明；
- 請務必只輸出純 JSON，不要任何解釋文字。
"""
            },
            {"role": "user", "content": f"請分析這段文字: {text}"}
        ]
    }

    res = requests.post(API_URL, headers=headers, json=data, timeout=90)
    res.raise_for_status()
    content = res.json()["choices"][0]["message"]["content"]

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{.*\}', content, re.S)
        if match:
            result = json.loads(match.group())
        else:
            raise ValueError(f"無法解析 JSON：{content}")
    return result


# === 🔹 分數檢查 ===
def validate_scores(result, link_id, article_id):
    for key in ["strategy_power", "strategy_emotion", "strategy_blame",
                "score_fear", "score_obligation", "score_guilt", "score_overall"]:
        if key in result:
            val = result[key]
            try:
                if not (0 <= float(val) <= 5):
                    write_log(f"⚠️ {key} 超出範圍 ({val})，已設為 None",
                              link_id=link_id, article_id=article_id)
                    result[key] = None
            except Exception:
                result[key] = None
    if "confidence" in result:
        try:
            if not (0 <= float(result["confidence"]) <= 1):
                write_log(f"⚠️ confidence 超出範圍 ({result['confidence']})，已設為 None",
                          link_id=link_id, article_id=article_id)
                result["confidence"] = None
        except Exception:
            result["confidence"] = None
    return result


# === 🔹 寫入分析結果 ===
def insert_analysis_result(conn, link_id, article_id, result):
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result = validate_scores(result, link_id, article_id)

    cur.execute("""
        INSERT INTO eb_evaluation (
            link_id, article_id, comment_id, level, created_at,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            model_name, model_version, knowledge_base
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        link_id, article_id, result.get("comment_id"),
        result.get("level", 1), created_at,
        result.get("strategy_power"),
        result.get("strategy_emotion"),
        result.get("strategy_blame"),
        result.get("score_fear"),
        result.get("score_obligation"),
        result.get("score_guilt"),
        result.get("pua_source"),
        result.get("main_strategy"),
        result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"),
        result.get("score_overall"),
        result.get("model_name", "mistral"),
        result.get("model_version", "latest"),
        result.get("knowledge_base", "pua_db")
    ))
    conn.commit()


# === 🔸 主程式 ===
def main():
    conn = sqlite3.connect(DB_FILE)
    ensure_column(conn, "articles", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "articles", "pua_status_reason", "TEXT")
    ensure_eb_evaluation_table(conn)
    ensure_column(conn, "eb_evaluation", "main_strategy_detail", "TEXT")

    df = get_pending_articles(conn, LIMIT_COUNT)
    if df.empty:
        write_log("⚠️ 沒有待分析資料。")
        conn.close()
        return

    for idx, row in df.iterrows():
        link_id = row["link_id"]
        article_id = row["id"]
        text = row["content"]

        try:
            result = analyze_text(text)
            insert_analysis_result(conn, link_id, article_id, result)
            update_article_status(conn, article_id, "done")
            write_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | overall={result.get('score_overall')} "
                f"| main={result.get('main_strategy')} | detail={result.get('main_strategy_detail')}",
                link_id=link_id, article_id=article_id, status="done"
            )
        except Exception as e:
            reason = f"{type(e).__name__}: {str(e)[:100]}"
            update_article_status(conn, article_id, "error", reason)
            write_log(f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | {reason}",
                      link_id=link_id, article_id=article_id, status="error")
            traceback.print_exc()

    conn.close()
    write_log("🎉 全部完成！")


# === 執行 ===
if __name__ == "__main__":
    main()

[2025-12-10 17:39:44] 🔎 欄位 pua_status 已存在，略過
[2025-12-10 17:39:44] 🔎 欄位 pua_status_reason 已存在，略過
[2025-12-10 17:39:44] ✅ 確認資料表 eb_evaluation 已存在
[2025-12-10 17:39:44] 🔎 欄位 main_strategy_detail 已存在，略過
[2025-12-10 17:39:44] 📘 共讀取 0 筆文章待分析（上限 0）
[2025-12-10 17:39:44] ⚠️ 沒有待分析資料。


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite")

df = pd.read_sql_query("""
    SELECT article_id, COUNT(*) AS cnt
    FROM eb_evaluation
    GROUP BY article_id
    HAVING cnt > 1;
""", conn)

print(df)
print("🔍 有重複筆數：", len(df))

conn.close()


                    article_id  cnt
0     68fce9b6af1137205015fd06    9
1     68fce9f3af11377624f11eda    8
2     68fcea01af11377624f11edb    9
3     68fcea6eaf11379f9c929394    9
4     68fcea81af11379f9c929395    9
...                        ...  ...
1038  6922a2a8dbeaa96e416e7d24    9
1039  6922a2b4dbeaa96e416e7d25    9
1040  6922a2c1dbeaa96e416e7d26   10
1041  6922a2cedbeaa96e416e7d27    9
1042  6922a3f6dbeaa96e416e7d28    4

[1043 rows x 2 columns]
🔍 有重複筆數： 1043


In [4]:
import re
import requests
import time
import json

LIMIT_COUNT =  5000
def analyze_article_and_comment(article_text, comment_text, comment_id=None):
    data = {
        "model": "mistral:latest",
        "knowledge": ["pua_db", "healthy_communication_db", "social_commentary_advice_db"],
        "messages": [
                    {
                        "role": "system",
                        "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
❗❗極度重要（請務必嚴格遵守，優先度最高）：
- 若評論中「沒有明確的操控、威脅、施壓、罪惡感誘發、恐懼引導、責任捆綁、指責、情緒綁架」，請將所有分數評為 0。
- 若評論內容屬於以下任一類型，也請評為 0：
  • 善意建議
  • 中立敘述
  • 理性討論
  • 情緒抒發但未要求對方負責
  • 一般社群語言、玩笑、生活分享
  • 批評但未包含控制或操作
- 除非評論中「明確存在」勒索策略，否則一律視為無勒索（即分數=0，main_strategy="none"）。
- 請勿因為語氣、情緒、抱怨、不滿，而推測或想像可能存在的勒索動機。
- 給出最低分（0）是最常見也是最正確的狀態。


📌 僅在出現「明確的施壓或操控語句」時，才可給 1~5 分。

分析規則：
- 本任務的分析主體是評論（comment）。
- 主文（article）僅提供上下文理解，不應被評分。
- 所有策略與分數僅針對評論內容進行判斷。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，若評論中完全沒有操控成分，請將所有分數設為 0。
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
"strategy_power": 0~5,
"strategy_emotion": 0~5,
"strategy_blame": 0~5,
"score_fear": 0~5,
"score_obligation": 0~5,
"score_guilt": 0~5,
"pua_source": "family／partner／friend／workplace／online／self",
"main_strategy": "power／emotion／blame／none",
"main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
"confidence": 0~1,
"score_overall": 0~5
}
                                """
                                                            },
                    {
                        "role": "user",
                        "content": f"""請分析以下內容，並將「評論(comment)」視為主要分析對象：

【主文（僅作為上下文，取300字）】
{article_text}

【評論（主要分析目標）】
{comment_text}""".strip()
                    }
                ]
    }
    print(data)

    res = requests.post(API_URL, headers=headers, json=data, timeout=90)
    print(res.status_code)
    print(res.text)
    content = res.json()["choices"][0]["message"]["content"]

    # 避免模型包文字 → 正常處理
    try:
        result = json.loads(content)
    except:
        match = re.search(r'\{.*\}', content, re.S)
        result = json.loads(match.group()) if match else {}

    # 自動補上 comment_id
    if comment_id and "comment_id" not in result:
        result["comment_id"] = comment_id

    return result

def get_pending_comments(conn, limit):
    query = f"""
    SELECT 
        c.id AS comment_id,
        c.comment_text AS comment_text,
        a.id AS article_id,
        a.link_id AS link_id,
        a.content AS article_text
    FROM article_comments c
    JOIN articles a 
        ON c.article_id = a.id
    WHERE c.comment_text IS NOT NULL
      AND TRIM(c.comment_text) != ''
      AND (c.pua_status IS NULL OR c.pua_status = 'pending')
    LIMIT {limit};
    """
    df = pd.read_sql_query(query, conn)
    write_log(f"📘 共讀取 {len(df)} 筆評論待分析（上限 {limit}）")
    return df


def update_comment_status(conn, comment_id, status, reason=None):
    cur = conn.cursor()
    cur.execute(
        "UPDATE article_comments SET pua_status = ?, pua_status_reason = ? WHERE id = ?;",
        (status, reason, comment_id)
    )
    conn.commit()
    

def insert_analysis_result(conn, link_id, article_id, result):
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result = validate_scores(result, link_id, article_id)

    cur.execute("""
        INSERT INTO eb_evaluation (
            link_id, article_id, comment_id, level, created_at,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            model_name, model_version, knowledge_base
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        link_id, article_id, result.get("comment_id"),
        result.get("level", 2), created_at,
        result.get("strategy_power"),
        result.get("strategy_emotion"),
        result.get("strategy_blame"),
        result.get("score_fear"),
        result.get("score_obligation"),
        result.get("score_guilt"),
        result.get("pua_source"),
        result.get("main_strategy"),
        result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"),
        result.get("score_overall"),
        result.get("model_name", "mistral"),
        result.get("model_version", "latest"),
        result.get("knowledge_base", "pua_db, healthy_communication_db, social_commentary_advice_db")
    ))
    conn.commit()


# === 🔸 主程式 ===
def main():
    conn = sqlite3.connect(DB_FILE)
    ensure_column(conn, "articles", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "articles", "pua_status_reason", "TEXT")
    ensure_eb_evaluation_table(conn)
    ensure_column(conn, "eb_evaluation", "main_strategy_detail", "TEXT")
    # ⭐ 新增：評論表也要有狀態欄位
    ensure_column(conn, "article_comments", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "article_comments", "pua_status_reason", "TEXT")

    df = get_pending_comments(conn, LIMIT_COUNT)
    if df.empty:
        write_log("⚠️ 沒有待分析評論。")
        conn.close()
        return

    for idx, row in df.iterrows():
        article_text = row["article_text"]
        comment_text = row["comment_text"]
        comment_id = row["comment_id"]
        article_id = row["article_id"]
        link_id = row["link_id"]

        try:
            result = analyze_article_and_comment(article_text, comment_text, comment_id)

            insert_analysis_result(conn, link_id, article_id, result)

            update_comment_status(conn, comment_id, "done")

            write_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | overall={result.get('score_overall')} "
                f"| main={result.get('main_strategy')} | detail={result.get('main_strategy_detail')}",
                link_id=link_id, article_id=article_id, comment_id=comment_id, status="done"
            )

        except Exception as e:
            reason = f"{type(e).__name__}: {str(e)[:100]}"
            update_comment_status(conn, comment_id, "error", reason)

            write_log(
                f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | {reason}",
                link_id=link_id, article_id=article_id, comment_id=comment_id, status="error"
            )
            traceback.print_exc()
        time.sleep(0.5)

    conn.close()
    write_log("🎉 全部完成！")


# === 執行 ===
if __name__ == "__main__":
    main()

[2025-12-11 08:46:05] 🔎 欄位 pua_status 已存在，略過
[2025-12-11 08:46:05] 🔎 欄位 pua_status_reason 已存在，略過
[2025-12-11 08:46:05] ✅ 確認資料表 eb_evaluation 已存在
[2025-12-11 08:46:05] 🔎 欄位 main_strategy_detail 已存在，略過
[2025-12-11 08:46:05] 🔎 欄位 pua_status 已存在，略過
[2025-12-11 08:46:05] 🔎 欄位 pua_status_reason 已存在，略過
[2025-12-11 08:46:05] 📘 共讀取 7 筆評論待分析（上限 5000）
{'model': 'mistral:latest', 'knowledge': ['pua_db', 'healthy_communication_db', 'social_commentary_advice_db'], 'messages': [{'role': 'system', 'content': '你是一個以繁體中文回覆的情緒勒索分析助手。\n❗❗極度重要（請務必嚴格遵守，優先度最高）：\n- 若評論中「沒有明確的操控、威脅、施壓、罪惡感誘發、恐懼引導、責任捆綁、指責、情緒綁架」，請將所有分數評為 0。\n- 若評論內容屬於以下任一類型，也請評為 0：\n  • 善意建議\n  • 中立敘述\n  • 理性討論\n  • 情緒抒發但未要求對方負責\n  • 一般社群語言、玩笑、生活分享\n  • 批評但未包含控制或操作\n- 除非評論中「明確存在」勒索策略，否則一律視為無勒索（即分數=0，main_strategy="none"）。\n- 請勿因為語氣、情緒、抱怨、不滿，而推測或想像可能存在的勒索動機。\n- 給出最低分（0）是最常見也是最正確的狀態。\n\n\n📌 僅在出現「明確的施壓或操控語句」時，才可給 1~5 分。\n\n分析規則：\n- 本任務的分析主體是評論（comment）。\n- 主文（article）僅提供上下文理解，不應被評分。\n- 所有策略與分數僅針對評論內容進行判斷。\n請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，

In [24]:
import sqlite3
import json
import pandas as pd

DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"

def get_comment_and_scores_as_dict(comment_id):
    conn = sqlite3.connect(DB_FILE)

    # 📝 抓評論
    df_comment = pd.read_sql_query("""
        SELECT 
            id AS comment_id,
            comment_text,
            article_id,
            link_id
        FROM article_comments
        WHERE id = ?
    """, conn, params=(comment_id,))

    if df_comment.empty:
        conn.close()
        return {"error": f"comment_id {comment_id} 不存在"}

    comment = df_comment.to_dict(orient="records")[0]

    # 📊 抓評分
    df_scores = pd.read_sql_query("""
        SELECT
            id AS eval_id,
            created_at,
            strategy_power,
            strategy_emotion,
            strategy_blame,
            score_fear,
            score_obligation,
            score_guilt,
            pua_source,
            main_strategy,
            main_strategy_detail,
            confidence,
            score_overall,
            model_name,
            model_version,
            knowledge_base
        FROM eb_evaluation
        WHERE comment_id = ?
        ORDER BY created_at ASC
    """, conn, params=(comment_id,))

    conn.close()

    scores = df_scores.to_dict(orient="records")

    return {
        "comment": comment,
        "evaluations": scores
    }


# 🧪 測試
if __name__ == "__main__":
    cid = "6923bf32af113796ec890f8b"   # ← 改成你的 comment_id
    data = get_comment_and_scores_as_dict(cid)
    print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "comment": {
    "comment_id": "6923bf32af113796ec890f8b",
    "comment_text": "我覺得既然有發現問題，可以去聽聽看心理諮商師的說法，或許會找到解法也不一定～如果擔心費用現在政府也有補助方案可以詢問看看！",
    "article_id": "68fce9b6af1137205015fd06",
    "link_id": "68f87edcaf1137700cb26e94"
  },
  "evaluations": [
    {
      "eval_id": 1410,
      "created_at": "2025-12-03 10:20:20",
      "strategy_power": 0.0,
      "strategy_emotion": 4.0,
      "strategy_blame": 3.0,
      "score_fear": 2.0,
      "score_obligation": 1.0,
      "score_guilt": 1.0,
      "pua_source": "family",
      "main_strategy": "emotion",
      "main_strategy_detail": "情緒操控，使用情感哀求、指責、壓力語言等方式來勒索",
      "confidence": 1.0,
      "score_overall": 3.0,
      "model_name": "mistral",
      "model_version": "latest",
      "knowledge_base": "pua_db, healthy_communication_db, social_commentary_advice_db"
    }
  ]
}
